In [1]:
from sentence_tokenization import *
from tf import Transformer
import numpy as np

In [2]:
max_sequence_length = 200
NEG_INFTY = -1e9

english_file = '/home/huang/Transformer-Neural-Network/dataset/ch/en-zh/UNv1.0.en-zh.en'
chinese_file = '/home/huang/Transformer-Neural-Network/dataset/ch/en-zh/UNv1.0.en-zh.zh'

START_TOKEN = '<START>'
PADDING_TOKEN = '<PADDING>'
END_TOKEN = '<END>'

english_vocabulary = [START_TOKEN, ' ', '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.',
                      '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9',
                      ':', '<', '=', '>', '?', '@',
                      'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L',
                      'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X',
                      'Y', 'Z',
                      '[', '\\', ']', '^', '_', '`',
                      'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l',
                      'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 
                      'y', 'z','?',
                      '{', '|', '}', '~', PADDING_TOKEN, END_TOKEN]

chinese_vocabulary = [START_TOKEN, ' ', '！', '“', "”", '#', '$', '%', '&', "’", "‘", '（', '）', '*', '+', 
                    '，', '-', '。', '/',  
                    '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '：', '<', '=', '>', '？', 
                    PADDING_TOKEN, END_TOKEN]

In [3]:
english_vocabulary, english_to_index, index_to_english = build_eng_vocab(english_file)
chinese_vocabulary, chinese_to_index, index_to_chinese = build_zh_vocab(chinese_file)

In [4]:
# 1. 初始化
english_embedding = SentenceEmbedding(
    max_sequence_length=200,
    d_model=512,
    language_to_index=english_to_index,
    language='en',
    START_TOKEN=START_TOKEN,
    END_TOKEN=END_TOKEN,
    PADDING_TOKEN=PADDING_TOKEN
)

chinese_embedding = SentenceEmbedding(
    max_sequence_length=200,
    d_model=512,
    language_to_index=chinese_to_index,
    language='zh',
    START_TOKEN=START_TOKEN,
    END_TOKEN=END_TOKEN,
    PADDING_TOKEN=PADDING_TOKEN
)

english_embedding = english_embedding.to(get_device())
chinese_embedding = chinese_embedding.to(get_device())

In [5]:
d_model = 512
batch_size = 32
ffn_hidden = 2048
num_heads = 8
drop_prob = 0.1
num_layers = 6
max_sequence_length = 200
zh_vocab_size = len(chinese_vocabulary)

transformer = Transformer(d_model, 
                          ffn_hidden,
                          num_heads, 
                          drop_prob, 
                          num_layers, 
                          max_sequence_length,
                          zh_vocab_size,
                          english_to_index,
                          chinese_to_index,
                          START_TOKEN, 
                          END_TOKEN, 
                          PADDING_TOKEN)

In [6]:
transformer

Transformer(
  (encoder): Encoder(
    (sentence_embedding): SentenceEmbedding(
      (embedding): Embedding(377146, 512)
      (position_encoder): PositionalEncoding()
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (layers): SequentialEncoder(
      (0): EncoderLayer(
        (attention): MutliHeadAttention(
          (qkv_layer): Linear(in_features=512, out_features=1536, bias=True)
          (linear_layer): Linear(in_features=512, out_features=512, bias=True)
        )
        (norm1): Normalnize()
        (dropout1): Dropout(p=0.1, inplace=False)
        (ffn): PostitionwiseFeedForward(
          (linear1): Linear(in_features=512, out_features=2048, bias=True)
          (linear2): Linear(in_features=2048, out_features=512, bias=True)
          (relu): ReLU()
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (norm2): Normalnize()
        (dropout2): Dropout(p=0.1, inplace=False)
      )
      (1): EncoderLayer(
        (attention): MutliHeadAttention(
  

In [7]:
# 2. 准备输入数据
with open(english_file, 'r') as file:
    english_sentences = file.readlines()
with open(chinese_file, 'r') as file:
    chinese_sentences = file.readlines()

# Limit Number of sentences ## 15886041
#                                100000
TOTAL_SENTENCES = 100000
english_sentences = english_sentences[:TOTAL_SENTENCES]
chinese_sentences = chinese_sentences[:TOTAL_SENTENCES]
english_sentences = [sentence.rstrip('\n') for sentence in english_sentences]
chinese_sentences = [sentence.rstrip('\n') for sentence in chinese_sentences]

valid_sentence_indicies = []
for index in range(len(chinese_sentences)):
    chinese_sentence, english_sentence = chinese_sentences[index], english_sentences[index]
    if is_valid_zh_length(chinese_sentence, max_sequence_length=max_sequence_length) \
        and is_valid_eng_length(english_sentence, max_sequence_length=max_sequence_length) \
        and is_valid_tokens(chinese_sentence, chinese_vocabulary):
            valid_sentence_indicies.append(index)
    
chinese_sentences = [chinese_sentences[i] for i in valid_sentence_indicies]
english_sentences = [english_sentences[i] for i in valid_sentence_indicies]


In [8]:
english_sentences[:3]

['RESOLUTION 918 (1994)',
 'Adopted by the Security Council at its 3377th meeting, on 17 May 1994',
 'The Security Council,']

In [9]:
chinese_sentences[:3]

['第918(1994)号决议', '1994年5月17日安全理事会第3377次会议通过', '安全理事会，']

In [10]:
# dataset = TextDataset()
dataset = TextDataset(english_sentences, chinese_sentences)

train_loader = DataLoader(dataset, batch_size)

In [11]:
from torch import nn

criterian = nn.CrossEntropyLoss(ignore_index=chinese_to_index[PADDING_TOKEN],
                                reduction='none')

# When computing the loss, we are ignoring cases when the label is the padding token
for params in transformer.parameters():
    if params.dim() > 1:
        nn.init.xavier_uniform_(params)

optim = torch.optim.Adam(transformer.parameters(), lr=1e-4)
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

In [12]:
def create_masks(eng_batch, zh_batch):
    num_sentences = len(eng_batch)
    look_ahead_mask = torch.full([max_sequence_length, max_sequence_length], True)
    look_ahead_mask = torch.triu(look_ahead_mask, diagonal=1)
    # diagonal=1 表示从主对角线偏移 1 的位置开始保留上三角
    encoder_padding_mask = torch.full([num_sentences, max_sequence_length, max_sequence_length], False)
    decoder_padding_mask_self_attention = torch.full([num_sentences, max_sequence_length, max_sequence_length], False)
    decoder_padding_mask_cross_attention = torch.full([num_sentences, max_sequence_length, max_sequence_length], False)

    for idx in range(num_sentences):
        eng_sentence_length, zh_sentence_length = len(eng_batch[idx]), len(zh_batch[idx])
        eng_chars_to_padding_mask = np.arange(eng_sentence_length + 1, max_sequence_length)
        zh_chars_to_padding_mask = np.arange(zh_sentence_length + 1, max_sequence_length)
        encoder_padding_mask[idx, :, eng_chars_to_padding_mask] = True
        encoder_padding_mask[idx, eng_chars_to_padding_mask, :] = True
        decoder_padding_mask_self_attention[idx, :, zh_chars_to_padding_mask] = True
        decoder_padding_mask_self_attention[idx, zh_chars_to_padding_mask, :] = True
        decoder_padding_mask_cross_attention[idx, :, eng_chars_to_padding_mask] = True
        decoder_padding_mask_cross_attention[idx, zh_chars_to_padding_mask, :] = True
    
    encoder_self_attention_mask = torch.where(encoder_padding_mask, NEG_INFTY, 0)
    decoder_self_attention_mask = torch.where(look_ahead_mask + decoder_padding_mask_self_attention, NEG_INFTY, 0)
    decoder_cross_attention_mask = torch.where(decoder_padding_mask_cross_attention, NEG_INFTY, 0)

    return encoder_self_attention_mask, decoder_self_attention_mask, decoder_cross_attention_mask

In [ ]:
transformer.train()
transformer.to(device)
total_loss = 0
num_epochs = 10

for epoch in range(num_epochs):
    print(f"Epoch {epoch}")
    iterator = iter(train_loader)
    for batch_num, batch in enumerate(iterator):
        transformer.train()
        eng_batch, zh_batch = batch
        encoder_self_attention_mask, decoder_self_attention_mask, decoder_cross_attention_mask = create_masks(eng_batch, zh_batch)
        optim.zero_grad()
        zh_predictions = transformer(eng_batch,
                                     zh_batch,
                                     encoder_self_attention_mask.to(device), 
                                     decoder_self_attention_mask.to(device), 
                                     decoder_cross_attention_mask.to(device),
                                     enc_start_token=False,
                                     enc_end_token=False,
                                     dec_start_token=True,
                                     dec_end_token=True)
        labels = transformer.decoder.sentence_embedding.batch_tokenize(zh_batch, start_token=False, end_token=True)
        loss = criterian(
            zh_predictions.view(-1, zh_vocab_size).to(device),
            labels.view(-1).to(device)
        ).to(device)
        valid_indicies = torch.where(labels.view(-1) == chinese_to_index[PADDING_TOKEN], False, True)
        loss = loss.sum() / valid_indicies.sum()
        loss.backward()
        optim.step()
        #train_losses.append(loss.item())
        if batch_num % 100 == 0:
            print(f"Iteration {batch_num} : {loss.item()}")
            print(f"English: {eng_batch[0]}")
            print(f"chinese Translation: {zh_batch[0]}")
            zh_sentence_predicted = torch.argmax(zh_predictions[0], axis=1)
            predicted_sentence = ""
            for idx in zh_sentence_predicted:
              if idx == chinese_to_index[END_TOKEN]:
                break
              predicted_sentence += index_to_chinese[idx.item()]
            print(f"chinese Prediction: {predicted_sentence}")


            transformer.eval()
            zh_sentence = ("",)
            eng_sentence = ("should we go to the house?",)
            for word_counter in range(max_sequence_length):
                encoder_self_attention_mask, decoder_self_attention_mask, decoder_cross_attention_mask= create_masks(eng_sentence, zh_sentence)
                predictions = transformer(eng_sentence,
                                          zh_sentence,
                                          encoder_self_attention_mask.to(device), 
                                          decoder_self_attention_mask.to(device), 
                                          decoder_cross_attention_mask.to(device),
                                          enc_start_token=False,
                                          enc_end_token=False,
                                          dec_start_token=True,
                                          dec_end_token=False)
                next_token_prob_distribution = predictions[0][word_counter] # not actual probs
                next_token_index = torch.argmax(next_token_prob_distribution).item()
                next_token = index_to_chinese[next_token_index]
                zh_sentence = (zh_sentence[0] + next_token, )
                if next_token == END_TOKEN:
                  break
            
            print(f"Evaluation translation (should we go to the house?) : {zh_sentence}")
            print("-------------------------------------------")

Epoch 0
Iteration 0 : 9.085518836975098
English: RESOLUTION 918 (1994)
chinese Translation: 第918(1994)号决议
chinese Prediction: 蜮痫痫字痫滏痫叵痫叵略滏痫叵滏痫滏略滏痫滏痫滏痫滏醅醅醅絶醅叵絶叵醅醅醅叵叵範醅醅醅醅醅醅醅醅醅醅叵昨痫百ç字痫字秧ç秧痫ç痫痫百ç字字痫秧痫ççç痫滏贾滏滏屆찈舛滏滏;敛滏舛滏屆찈滏滏蓖滏舛滏靳滏滏蓖昨蓖潜锢潜潜潜滏滏埚潜潜潜滏潞锢埚滏锢潜ФФ滏潜舛舛叵數叵叵武叵叵叵叵叵叵叵．叵叵叵叵叵舛府叵叵舛逹ç遑戋烩贾遑ç閚遑烩遑逹閚遑遑烩烩逹遑遑되遑遑遑叵叵叵叵叵叵叵叵叵滏叵滏叵叵叵叵叵滏叵叵滏滏叵叵叵
